In [2]:
!pip install torch_optimizer
!pip install medpy
!pip install albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for medpy: filename=medpy-0.5.2-py3-none-any.whl size=224805 sha256=75bc403be24a8a550dc2bd2564aacdd41b16d5189ebbf7030a6ee31d8024c51e
  Stored in directory: /root/.cache/pip/wheels/89/5a/f8/b3def53b9c2133d2f8698ea2173bb5df63bd8e761ce8e9aec9
Successfully built medpy


In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
from PIL import Image
from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib.patches as mpatches

In [4]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torchvision.transforms.functional as TF
import random


class SynapseDataset(Dataset):
    def __init__(self, npz_files, augment=False, image_size=(112, 112)):
        self.files = []

        for f in npz_files:
            npz = np.load(f)
            image, label = npz['image'], npz['label']

            if np.max(image) > 0 and np.max(label) > 0:
                self.files.append(f)

        self.augment = augment
        self.image_size = image_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        npz = np.load(self.files[idx])

        image = npz['image'].astype(np.float32)
        label = npz['label'].astype(np.int64)

        # Min-max normalization
        image = (image - image.min()) / (
            image.max() - image.min() + 1e-8
        )

        image = np.expand_dims(image, axis=0)

        image = torch.from_numpy(image)
        label = torch.from_numpy(label)

        # Resize
        image = TF.resize(image, self.image_size)

        label = TF.resize(
            label.unsqueeze(0).float(),
            self.image_size,
            interpolation=TF.InterpolationMode.NEAREST
        ).squeeze(0).long()

        # Slightly stronger augmentation
        if self.augment:
            image, label = self.random_augment(image, label)

        return image, label

    def random_augment(self, image, label):

        # Horizontal flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            label = TF.hflip(label)

        # Slightly larger rotation
        angle = random.uniform(-10, 10)

        image = TF.rotate(
            image,
            angle,
            interpolation=TF.InterpolationMode.BILINEAR
        )

        label = TF.rotate(
            label.unsqueeze(0),
            angle,
            interpolation=TF.InterpolationMode.NEAREST
        ).squeeze(0)

        # Slight brightness variation
        if random.random() > 0.5:
            factor = random.uniform(0.90, 1.10)
            image = TF.adjust_brightness(image, factor)

        # Slight contrast variation
        if random.random() > 0.5:
            factor = random.uniform(0.90, 1.10)
            image = TF.adjust_contrast(image, factor)

        return image, label

In [5]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

dataset_path = "/kaggle/input/datasets/dogcdt/synapse/Synapse/train_npz"
all_npz_files = sorted([os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith('.npz')])

# Split
train_files, val_files = train_test_split(all_npz_files, test_size=0.25, random_state=42)

train_dataset = SynapseDataset(train_files)
val_dataset = SynapseDataset(val_files)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4)

In [6]:
images, masks = next(iter(train_loader))

print("Image batch shape:", images.shape) 
print("Mask batch shape:", masks.shape)   

Image batch shape: torch.Size([4, 1, 112, 112])
Mask batch shape: torch.Size([4, 112, 112])


In [7]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights


class ResSkip(nn.Module):
    def __init__(self, res_ch, out_ch, res_up=1):
        super().__init__()
        self.res_conv = nn.Conv2d(res_ch, out_ch, 3, padding=1)
        self.res_deconvs = nn.ModuleList([nn.ConvTranspose2d(out_ch, out_ch, 2, 2) for _ in range(res_up)])

    def forward(self, res_feat):
        r = self.res_conv(res_feat)
        for de in self.res_deconvs:
            r = de(r)
        return r


def _match_add(a, b):
    min_h = min(a.shape[2], b.shape[2])
    min_w = min(a.shape[3], b.shape[3])
    a = a[:, :, :min_h, :min_w]
    b = b[:, :, :min_h, :min_w]
    return a + b


class SegmentationModel(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.DEFAULT)

        old_conv1 = backbone.conv1
        backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv1.out_channels,
            kernel_size=old_conv1.kernel_size,
            stride=old_conv1.stride,
            padding=old_conv1.padding,
            bias=False
        )
        with torch.no_grad():
            backbone.conv1.weight[:] = old_conv1.weight.mean(dim=1, keepdim=True)

        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # 256 ch
        self.layer2 = backbone.layer2   # 512 ch
        self.layer3 = backbone.layer3   # 1024 ch
        self.layer4 = backbone.layer4   # 2048 ch

        self.bottleneck = nn.Sequential(
            nn.ConvTranspose2d(2048, 2048, 2, 2),
            nn.Conv2d(2048, 768, 3, padding=1),
            nn.BatchNorm2d(768),
            nn.ReLU(inplace=True),
            nn.Conv2d(768, 768, 3, padding=1),
            nn.BatchNorm2d(768),
            nn.ReLU(inplace=True),
        )

        # Decoder
        self.up1 = nn.ConvTranspose2d(768, 512, 2, 2)
        self.up2 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.up4 = nn.ConvTranspose2d(128,  64, 2, 2)

        self.skip1 = ResSkip(res_ch=1024, out_ch=512, res_up=1)
        self.skip2 = ResSkip(res_ch=512,  out_ch=256, res_up=1)
        self.skip3 = ResSkip(res_ch=256,  out_ch=128, res_up=1)

        self.input_skip = nn.Conv2d(1, 64, 3, padding=1)

        self.final_conv = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        l0 = self.stem(x)      
        l1 = self.layer1(l0)   
        l2 = self.layer2(l1)   
        l3 = self.layer3(l2)     
        l4 = self.layer4(l3)   

        b = self.bottleneck(l4)

        up1 = _match_add(self.up1(b),   self.skip1(l3))
        up2 = _match_add(self.up2(up1), self.skip2(l2))
        up3 = _match_add(self.up3(up2), self.skip3(l1))
        up4 = _match_add(self.up4(up3), self.input_skip(x))

        out = self.final_conv(up4)                        
        return out

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SegmentationModel(num_classes=9).to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


In [9]:
import torch
import torch.nn.functional as F
import numpy as np
from medpy.metric.binary import hd

def dice_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    intersection = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    union = preds_onehot.sum(dim=(0,2,3)) + targets_onehot.sum(dim=(0,2,3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean().item()

def iou_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    intersection = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    union = (preds_onehot + targets_onehot - preds_onehot*targets_onehot).sum(dim=(0,2,3))
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

def pixel_accuracy(preds, targets):
    return (preds.argmax(dim=1) == targets).float().mean().item()

def precision_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    tp = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    fp = (preds_onehot * (1 - targets_onehot)).sum(dim=(0,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    return precision.mean().item()

def recall_score(preds, targets, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    tp = (preds_onehot * targets_onehot).sum(dim=(0,2,3))
    fn = ((1 - preds_onehot) * targets_onehot).sum(dim=(0,2,3))
    recall = (tp + eps) / (tp + fn + eps)
    return recall.mean().item()

def f1_score(preds, targets, eps=1e-7):
    prec = precision_score(preds, targets, eps)
    rec = recall_score(preds, targets, eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    return f1

def hausdorff_distance(preds, targets):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=preds.shape[1]).permute(0,3,1,2).float()
    targets_onehot = F.one_hot(targets, num_classes=preds.shape[1]).permute(0,3,1,2).float()
    distances = []
    for b in range(preds_onehot.shape[0]):
        for c in range(preds_onehot.shape[1]):
            p = preds_onehot[b,c].cpu().numpy().astype(bool)
            t = targets_onehot[b,c].cpu().numpy().astype(bool)
            if p.sum() == 0 or t.sum() == 0:
                distances.append(np.nan)
            else:
                try:
                    distances.append(hd(p, t))
                except:
                    distances.append(np.nan)
    return np.nanmean(distances)


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, preds, targets):
        num_classes = preds.shape[1]
        preds = F.softmax(preds, dim=1)
        preds_one_hot = F.one_hot(preds.argmax(dim=1), num_classes=num_classes).permute(0,3,1,2).float()
        targets_one_hot = F.one_hot(targets, num_classes=num_classes).permute(0,3,1,2).float()

        intersection = (preds_one_hot * targets_one_hot).sum(dim=(0,2,3))
        union = preds_one_hot.sum(dim=(0,2,3)) + targets_one_hot.sum(dim=(0,2,3))
        dice = (2. * intersection + self.eps) / (union + self.eps)
        return 1 - dice.mean()

ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

def combined_loss(outputs, targets, ce_weight=0.5, dice_weight=0.5):
    return ce_weight * ce_loss(outputs, targets) + dice_weight * dice_loss(outputs, targets)

criterion = combined_loss
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5,weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=4, factor=0.5)

In [11]:
from torch.amp import autocast, GradScaler
import torch
import torch.nn.functional as F
from tqdm import tqdm

scaler = GradScaler() 
accumulation_steps = 4
num_epochs =100

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0.0
    train_dice = train_acc = train_iou = train_hd = 0.0
    train_mean_iou = train_precision = train_recall = train_f1 = 0.0

    optimizer.zero_grad()

    for step, (images, masks) in tqdm(enumerate(train_loader)):
        images, masks = images.to(device), masks.to(device)

        with autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)  # [B, C, H, W]
            loss = criterion(outputs, masks) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * accumulation_steps  
        train_dice += dice_score(outputs, masks)
        train_acc += pixel_accuracy(outputs, masks)
        train_iou += iou_score(outputs, masks)
        train_hd += hausdorff_distance(outputs, masks)
        train_mean_iou += iou_score(outputs, masks)  
        train_precision += precision_score(outputs, masks)
        train_recall += recall_score(outputs, masks)
        train_f1 += f1_score(outputs, masks)

    n_train = len(train_loader)
    avg_train_loss = train_loss / n_train
    avg_train_dice = train_dice / n_train
    avg_train_acc = train_acc / n_train
    avg_train_iou = train_iou / n_train
    avg_train_hd = train_hd / n_train
    avg_train_mean_iou = train_mean_iou / n_train
    avg_train_precision = train_precision / n_train
    avg_train_recall = train_recall / n_train
    avg_train_f1 = train_f1 / n_train

    model.eval()
    val_loss = 0.0
    val_dice = val_acc = val_iou = val_hd = 0.0
    val_mean_iou = val_precision = val_recall = val_f1 = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader):
            images, masks = images.to(device), masks.to(device)
            with autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, masks)

            val_loss += loss.item()
            val_dice += dice_score(outputs, masks)
            val_acc += pixel_accuracy(outputs, masks)
            val_iou += iou_score(outputs, masks)
            val_hd += hausdorff_distance(outputs, masks)
            val_mean_iou += iou_score(outputs, masks)
            val_precision += precision_score(outputs, masks)
            val_recall += recall_score(outputs, masks)
            val_f1 += f1_score(outputs, masks)

    n_val = len(val_loader)
    avg_val_loss = val_loss / n_val
    avg_val_dice = val_dice / n_val
    avg_val_acc = val_acc / n_val
    avg_val_iou = val_iou / n_val
    avg_val_hd = val_hd / n_val
    avg_val_mean_iou = val_mean_iou / n_val
    avg_val_precision = val_precision / n_val
    avg_val_recall = val_recall / n_val
    avg_val_f1 = val_f1 / n_val

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch:2d} | "
          f"Train Loss: {avg_train_loss:.6f} | Dice: {avg_train_dice:.4f} | Acc: {avg_train_acc:.4f} | "
          f"IoU: {avg_train_iou:.4f} | mIoU: {avg_train_mean_iou:.4f} | Precision: {avg_train_precision:.4f} | Recall: {avg_train_recall:.4f} | F1: {avg_train_f1:.4f} | HD: {avg_train_hd:.4f} || "
          f"Val Loss: {avg_val_loss:.6f} | Dice: {avg_val_dice:.4f} | Acc: {avg_val_acc:.4f} | "
          f"IoU: {avg_val_iou:.4f} | mIoU: {avg_val_mean_iou:.4f} | Precision: {avg_val_precision:.4f} | Recall: {avg_val_recall:.4f} | F1: {avg_val_f1:.4f} | HD: {avg_val_hd:.4f}")

256it [00:27,  9.22it/s]
100%|██████████| 65/65 [00:03<00:00, 16.60it/s]

Epoch  1 | Train Loss: 1.444346 | Dice: 0.0753 | Acc: 0.4051 | IoU: 0.0605 | mIoU: 0.0605 | Precision: 0.3220 | Recall: 0.2155 | F1: 0.2364 | HD: 49.5913 || Val Loss: 1.311723 | Dice: 0.1234 | Acc: 0.8860 | IoU: 0.1167 | mIoU: 0.1167 | Precision: 0.4668 | Recall: 0.2117 | F1: 0.2671 | HD: 42.6102



256it [00:18, 13.60it/s]
100%|██████████| 65/65 [00:02<00:00, 22.53it/s]

Epoch  2 | Train Loss: 1.027659 | Dice: 0.2135 | Acc: 0.9210 | IoU: 0.2092 | mIoU: 0.2092 | Precision: 0.8765 | Recall: 0.2385 | F1: 0.3510 | HD: 48.0008 || Val Loss: 0.743105 | Dice: 0.2093 | Acc: 0.9250 | IoU: 0.2053 | mIoU: 0.2053 | Precision: 0.9917 | Recall: 0.2137 | F1: 0.3323 | HD: 51.2385



256it [00:18, 13.99it/s]
100%|██████████| 65/65 [00:02<00:00, 22.48it/s]

...

Epoch 98 | Train Loss: 0.118853 | Dice: 0.7907 | Acc: 0.9899 | IoU: 0.7263 | mIoU: 0.7263 | Precision: 0.8962 | Recall: 0.7946 | F1: 0.8393 | HD: 4.5111 || Val Loss: 0.135188 | Dice: 0.7651 | Acc: 0.9873 | IoU: 0.6934 | mIoU: 0.6934 | Precision: 0.8992 | Recall: 0.7536 | F1: 0.8176 | HD: 5.5993



256it [00:23, 10.86it/s]
100%|██████████| 65/65 [00:04<00:00, 14.83it/s]

Epoch 99 | Train Loss: 0.119064 | Dice: 0.7900 | Acc: 0.9900 | IoU: 0.7246 | mIoU: 0.7246 | Precision: 0.8957 | Recall: 0.7918 | F1: 0.8380 | HD: 4.4382 || Val Loss: 0.135845 | Dice: 0.7636 | Acc: 0.9873 | IoU: 0.6898 | mIoU: 0.6898 | Precision: 0.8903 | Recall: 0.7513 | F1: 0.8120 | HD: 5.5565



256it [00:23, 10.90it/s]
100%|██████████| 65/65 [00:04<00:00, 14.84it/s]

Epoch 100 | Train Loss: 0.120210 | Dice: 0.7873 | Acc: 0.9901 | IoU: 0.7225 | mIoU: 0.7225 | Precision: 0.8981 | Recall: 0.7906 | F1: 0.8385 | HD: 4.4583 || Val Loss: 0.131522 | Dice: 0.7719 | Acc: 0.9875 | IoU: 0.6986 | mIoU: 0.6986 | Precision: 0.8967 | Recall: 0.7592 | F1: 0.8195 | HD: 5.5176


In [ ]:
import torch
import torch.nn.functional as F

def dice_per_class(preds, targets, num_classes, eps=1e-7):
    preds_onehot = F.one_hot(preds.argmax(dim=1), num_classes=num_classes) 
    preds_onehot = preds_onehot.permute(0,3,1,2).float() 

    targets_onehot = F.one_hot(targets, num_classes=num_classes)  
    targets_onehot = targets_onehot.permute(0,3,1,2).float()  

    dice_scores = []
    for c in range(num_classes):
        pred_c = preds_onehot[:,c]
        target_c = targets_onehot[:,c]
        intersection = (pred_c * target_c).sum(dim=(0,1,2))
        union = pred_c.sum(dim=(0,1,2)) + target_c.sum(dim=(0,1,2))
        dice = (2*intersection + eps)/(union + eps)
        dice_scores.append(dice.item())
    
    return dice_scores  

In [ ]:
model.eval()
num_classes = 9  # 0:background + 8 organs

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)  # [B, C, H, W]
        per_class_dice = dice_per_class(outputs, masks, num_classes=num_classes)
        break  # only first batch for demo

organ_names = ["Background", "Aorta", "Gallbladder", "Left Kidney", "Right Kidney", 
               "Liver", "Pancreas", "Spleen", "Stomach"]

for name, dice in zip(organ_names, per_class_dice):
    print(f"{name}: {dice*100:.2f}%")